# Distillation Offset-Free MPC Baseline

This notebook keeps the unified baseline MPC runner and plotting stack, but its default setpoints, run lengths, and controller settings are aligned with the archived distillation baseline notebooks (`MPCOffsetFree*`). Use it when you want the canonical distillation baseline saved into `Distillation/Data` while keeping the old study setup.

In [ ]:
from pathlib import Path
import os
import pickle

from utils.notebook_setup import prepare_distillation_notebook_env, print_notebook_summary

# Main notebook controls.
# Keep these close to the archived baseline notebooks unless you are intentionally changing the study.
RUN_MODE = "nominal"  # "nominal" | "disturb"
DISTURBANCE_PROFILE = "none"  # "none" | "ramp" | "fluctuation"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Aspen selection. Use the preset for the legacy plant file or set manual overrides below.
ASPEN_PRESET = "default"
ASPEN_PATH_OVERRIDE = None
SNAPS_PATH_OVERRIDE = None
ASPEN_ROOT_OVERRIDE = None
DISTILLATION_VISIBLE = True

# Canonical data/result directories. Leave as None to use Distillation/Data and Distillation/Results.
DISTILLATION_DATA_DIR_OVERRIDE = None
DISTILLATION_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides for saved outputs.
RESULT_PREFIX_OVERRIDE = None
BASELINE_SAVE_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to keep the legacy-aligned defaults.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(
    run_mode=RUN_MODE,
    disturbance_profile=DISTURBANCE_PROFILE,
    family="baseline",
    aspen_preset=ASPEN_PRESET,
    dyn_path_override=ASPEN_PATH_OVERRIDE,
    snaps_path_override=SNAPS_PATH_OVERRIDE,
    aspen_root_override=ASPEN_ROOT_OVERRIDE,
    data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE,
    results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

print_notebook_summary(
    "Distillation baseline configuration",
    {
        "Distillation data dir": DATA_DIR,
        "Distillation results": RESULT_DIR,
        "Run mode": RUN_MODE,
        "Disturbance profile": DISTURBANCE_PROFILE,
        "Aspen source": ASPEN_SOURCE,
        "Aspen dynf": DYN_PATH,
        "Aspen snaps": SNAPS_PATH,
    },
)


In [ ]:
import numpy as np

from Simulation.mpc import MpcSolverGeneral
from utils.helpers import apply_min_max
from utils.mpc_baseline_runner import run_offsetfree_mpc
from utils.observer import compute_observer_gain
from utils.plotting import plot_baseline_mpc_results
from utils.rewards import make_reward_fn_relative_QR
from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_BASELINE_RUN_PROFILES,
    DISTILLATION_BASELINE_SETPOINTS_PHYS,
    DISTILLATION_INPUT_BOUNDS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_OBSERVER_POLES,
    DISTILLATION_SETPOINT_RANGE_PHYS,
    DISTILLATION_SS_INPUTS,
    RL_REWARD_DEFAULTS,
)
from systems.distillation.data_io import canonical_baseline_path, load_distillation_system_data
from systems.distillation.labels import DISTILLATION_SYSTEM_METADATA
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule


In [ ]:
# Build the distillation plant and load the canonical model/scaling bundle.
# The setpoint range below matches the old baseline notebooks and is kept separate from the RL horizon pair.
nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS.copy()
u_min = DISTILLATION_INPUT_BOUNDS["u_min"].copy()
u_max = DISTILLATION_INPUT_BOUNDS["u_max"].copy()
setpoint_y = DISTILLATION_SETPOINT_RANGE_PHYS.copy()

system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=DELTA_T_HOURS,
    visible=DISTILLATION_VISIBLE,
)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario_phys = DISTILLATION_BASELINE_SETPOINTS_PHYS.copy()
y_sp_scenario = (
    apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:])
    - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
)

print_notebook_summary(
    "Resolved distillation plant paths",
    {
        "Data dir": DATA_DIR,
        "Results dir": RESULT_DIR,
        "Aspen dynf": DYN_PATH,
        "Aspen snaps": SNAPS_PATH,
    },
)


In [ ]:
# User-editable controller defaults.
poles = DISTILLATION_OBSERVER_POLES.copy()
L = compute_observer_gain(A_aug, C_aug, poles)

n_inputs = int(B_aug.shape[1])
predict_h = 6
cont_h = 3
Q1_penalty = 1.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

RUN_PROFILE = DISTILLATION_BASELINE_RUN_PROFILES[(RUN_MODE, DISTURBANCE_PROFILE)]
n_tests = RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
warm_start = RUN_PROFILE["warm_start"]
TEST_CYCLE = list(RUN_PROFILE["test_cycle"]) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or f"distillation_baseline_{RUN_MODE}_{DISTURBANCE_PROFILE}_unified"
BASELINE_SAVE_PATH = Path(BASELINE_SAVE_PATH_OVERRIDE).expanduser() if BASELINE_SAVE_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE)

disturbance_schedule = build_distillation_disturbance_schedule(
    RUN_MODE,
    DISTURBANCE_PROFILE,
    total_steps=n_tests * set_points_len * len(y_sp_scenario_phys),
)
reward_defaults = dict(RL_REWARD_DEFAULTS)
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs,
    reward_defaults["k_rel"],
    reward_defaults["band_floor_phys"],
    reward_defaults["Q_diag"],
    reward_defaults["R_diag"],
    tau_frac=reward_defaults["tau_frac"],
    gamma_out=reward_defaults["gamma_out"],
    gamma_in=reward_defaults["gamma_in"],
    beta=reward_defaults["beta"],
    gate=reward_defaults["gate"],
    lam_in=reward_defaults["lam_in"],
    bonus_kind=reward_defaults["bonus_kind"],
    bonus_k=reward_defaults["bonus_k"],
    bonus_p=reward_defaults["bonus_p"],
    bonus_c=reward_defaults["bonus_c"],
)
MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)
print_notebook_summary(
    "Resolved baseline parameters",
    {
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "plot_start_episode": PLOT_START_EPISODE,
        "legacy_baseline_setpoints_phys": y_sp_scenario_phys.tolist(),
        "baseline_save_path": BASELINE_SAVE_PATH,
    },
)


In [ ]:
mpc_cfg = {
    "run_mode": RUN_MODE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": float(nominal_conditions[0]),
    "nominal_qs": 0.0,
    "nominal_ha": 1.0,
    "qi_change": 1.0,
    "qs_change": 1.0,
    "ha_change": 1.0,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
}

runtime_ctx = {
    "system": system,
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "L": L,
    "disturbance_schedule": disturbance_schedule,
    "system_stepper": distillation_system_stepper,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA["disturbance_labels"],
}

result_bundle = run_offsetfree_mpc(mpc_cfg=mpc_cfg, runtime_ctx=runtime_ctx)
with open(BASELINE_SAVE_PATH, "wb") as handle:
    pickle.dump(result_bundle, handle)
print(f"Saved canonical baseline to: {BASELINE_SAVE_PATH}")


In [ ]:
out_dir = plot_baseline_mpc_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)
print(f"Baseline plot/output directory: {out_dir}")

try:
    system.close(SNAPS_PATH)
except Exception as exc:
    print(f"Could not close Aspen cleanly: {exc}")
